# Lecture 8: GIS automation and spatial analysis as a process

> **🙏 Acknowledgements**
>
> This self-study lecture is based on the guest lecture *GIS automation & spatial analysis as a process* by **Tatu Leppämäki** (Doctoral Researcher, [Digital Geography Lab](https://www.helsinki.fi/en/researchgroups/digital-geography-lab), University of Helsinki), given on this course in November 2025. The shade map story, the model screenshots, the automation plot and the practical map are Tatu's. The text has been extended for self-study. Thank you, Tatu!

**How to use this page**

This lecture replaces a classroom session, so take your time with it. Reading it through takes around 45–60 minutes. Along the way you will find short 💭 *Pause and think* questions. You don't need to hand in answers to them, but they are a good way to check that the ideas make sense to you before you start Practical 8. There is a short self-check at the end.

**Learning objectives**

After this lecture you should be able to:

- describe GIS analysis as a process with stages, using the PPDAC model;
- explain why that way of thinking helps when you plan, carry out and report an analysis;
- explain what we mean by automation in GIS, and what it is not;
- name the main ways to automate work in QGIS: batch processing, the Graphical Modeler and Python;
- judge when automating a task is worth the effort.

## 1. A question from the newspaper

In June 2025, *Helsingin Sanomat* ran a summer story on [which bar terraces in central Helsinki get the most sun](https://www.hs.fi/helsinki/art-2000011290791.html). The paper used [Shademap](https://shademap.app/), a commercial web tool that simulates shadows from buildings, terrain and trees for any time and date.

Tatu read the article and had a simple thought: *surely it wouldn't be too hard to calculate shade maps with open data and open tools?*

Before opening any software, he needed three things:

1. **A goal.** Here, a slightly unscientific one: *I want to know where I can get a beer in the sun.*
2. **The right tools.** QGIS, with the **r.sun** solar radiation model from GRASS GIS, which you can run from the QGIS Processing Toolbox (the variant used here is called *r.sun.incidout*).
3. **Data.** A raster of surface heights for the city, plus the **slope** and **aspect** rasters derived from it. Slope tells how steep each cell is and aspect which way it faces, and both affect how sunlight hits a surface.

<img src="figures/rsun_dialog.png" width="480">

<img src="figures/dsm_slope.png" width="260">

*Figure 1. The r.sun.incidout tool in QGIS (top), and the height model and slope raster used as inputs (bottom). Figures: Tatu Leppämäki.*

> **Note: DEM or DSM?**
>
> A *digital elevation model* (DEM) usually means the bare ground surface. A *digital surface model* (DSM) also includes buildings and trees. For shadows in a city you need the DSM, because the buildings cast the shadows. That is the raster in Figure 1: you can see the building blocks and individual tree crowns. In Finland, a DSM can be produced from the airborne laser scanning (lidar) data that the National Land Survey publishes as open data.

**What does r.sun actually do?** For a given date, time and location, it calculates where the sun is in the sky. For every raster cell it then checks two things: whether something taller (a building, a tree, a hill) blocks the line towards the sun, and at what angle the sun's rays hit the surface. The second part depends on the slope and aspect of the cell. Cells where the line to the sun is blocked are in shadow. The model is described in Hofierka & Šúri (2002).

After some work, the result looked like this:

<img src="figures/shade_result.jpg" width="520">

*Figure 2. Calculated shadows (grey) around Senate Square and Helsinki Cathedral at one moment in time. Figure: Tatu Leppämäki, background map © OpenStreetMap contributors.*

It works! But there were two problems:

1. A shade map is valid for **one moment only**. To say anything useful about a terrace, you need to run it for many times of day and several days of the year: tens of runs.
2. **Each run took about 30 minutes.**

Do the maths. Say you want every hour from 10 to 21 on four dates over the summer. That is 48 runs, or roughly 24 hours of computing. And someone has to type in the date and time parameters 48 times without a single mistake.

This is where **automation** comes in. But before we automate anything, it helps to look at the work itself.

## 2. GIS analysis as a process: the PPDAC model

When you read Tatu's story again, you can see that he went through a sequence of stages: he had a question, worked out how he could answer it, found data, ran the analysis, and looked at whether the result answered the question. This sequence has a name.

The **PPDAC** model comes from statistics. MacKay & Oldford (2000) used it to describe how an empirical investigation proceeds, and de Smith, Goodchild & Longley adopted it in their textbook *Geospatial Analysis* as a framework for spatial analysis. It has five stages:

<img src="figures/ppdac_cycle.png" width="480">

*Figure 3. The PPDAC cycle. Each stage has one guiding question.*

| Stage | Guiding question | In the shade map example |
|---|---|---|
| **Problem** | What do I want to find out? | Which terraces in central Helsinki are in the sun, and when? |
| **Plan** | How can I do this? | Simulate shadows from a surface model for selected dates and times, then compare with terrace locations. |
| **Data** | What data do I need, and where from? | An open DSM from laser scanning; slope and aspect derived from it; terrace locations. |
| **Analysis** | Which software and methods do I need? | QGIS + GRASS r.sun, then cleaning up the result raster. |
| **Conclusions** | What have I learned? Did I answer the problem? | A shade map per date and time; check against reality before trusting it. |

### It is a cycle, not a checklist

The arrows in Figure 3 go both ways for a reason. Real projects rarely run straight from Problem to Conclusions. Some typical reasons to go back a step:

- the data you planned to use does not exist, is not open, or is too coarse, so you go back to the **Plan**;
- the analysis shows that the data has gaps or errors, so you go back to **Data**;
- the conclusions raise a new question, so a new cycle starts from a new **Problem**.

Tatu's own project is a good example. Once the tool worked, the question could easily be turned around: instead of *where can I sit in the sun?*, ask *where are the shadiest places in Helsinki?* On a hot summer day that is arguably the more useful question, for example when thinking about heat and urban planning. Same data, same model, new problem.

### Why bother thinking in stages?

- **It forces you to state the problem first.** Many weak GIS projects start from "I have this dataset, let's map it". Starting from a question makes it much easier to choose data and methods.
- **It makes your choices visible.** Every stage involves decisions: which data, which buffer distance, which classification. Writing them down lets others (and you, later) check them.
- **It gives your report a structure.** This is why Practical 8 asks you to structure your report with PPDAC.
- **It is the first step towards automation.** Once you have written down the Plan and Analysis stages as clear, ordered steps, you have more or less described a model that a computer can run.

> 💭 **Pause and think**
>
> Pick one of the earlier practicals on this course, for example the hotspot analysis (Practical 5) or the segregation analysis (Practical 4). Try to describe it in PPDAC terms. What was the problem? Which steps were the plan? Was there a moment where you had to go back a stage?

## 3. What do we mean by automation?

In this lecture, **automating a GIS task** means writing down a sequence of steps once, so that the computer can run them again, with different input data or parameter values, without you clicking through every tool by hand.

You may have heard a lot recently about AI "agents" that could do GIS work on their own. Li et al. (2025) describe six **levels of automation** for GIS, from fully manual work to fully autonomous systems:

| Level | Name | Who does what |
|---|---|---|
| 0 | Manual | The person does everything: collects data, runs each tool, makes the maps. |
| **1** | **Routine-aware** | **The person prepares data and defines the workflow; the computer runs that predefined workflow for repetitive tasks.** |
| 2 | Workflow-aware | The system can build and run a workflow for given data; the person sets inputs and interprets the results. |
| 3 | Data-aware | The system can also find and prepare suitable data itself. |
| 4 | Result-aware | The system checks its own results and keeps refining the workflow and data. |
| 5 | Knowledge-aware | Fully autonomous; the system learns from earlier successes and failures. |

*Table summarised from Li et al. (2025).*

Could an AI agent do the job of a GIS analyst today? Tatu's answer was *not for now*, and research on the higher levels is still at an early stage. Even where AI tools help with parts of the work, someone still has to judge whether the data fits the question and whether the result makes sense.

**This week is about Level 1.** You stay in charge of every decision, and the computer does the repetitive part.

## 4. Why automate?

Tatu gave five reasons. They are all worth taking seriously:

- **Repetitive tasks.** Running the same steps for 48 date-and-time combinations, 20 municipalities or 10 years of data.
- **Long workflows.** If an analysis has ten steps, every manual run gives you ten chances to forget something.
- **Iterating over a parameter.** Results often depend on a choice you made, such as a buffer distance of 10, 100 or 1000 m. Trying several values and comparing them (a simple *sensitivity analysis*) is good practice, and it is much easier when the workflow is automated.
- **Sharing your work, or coming back to it later.** A saved model shows exactly what was done, in which order, with which settings. Six months from now, you will be grateful for it.
- **Fewer human errors.** No misclicks, no wrong layer picked from a drop-down, no forgotten step.

There is one more reason that is easy to overlook: **a model is documentation.** When someone asks how you got your result, you can show them the model.

> **A word of caution**
>
> Automation repeats mistakes just as reliably as it repeats correct steps. If the first step uses the wrong field or the wrong coordinate system, every run will be wrong. Always test a new model on a small piece of data and look at the intermediate results before you trust it.

## 5. How to automate in QGIS

QGIS has several ways to automate work. They range from "no code at all" to full programming.

### 5.1 Batch processing: one tool, many runs

Almost every tool in the Processing Toolbox has a **Run as Batch Process…** button at the bottom of its dialog. It opens a table where each row is one run of the tool. You can fill in different input layers or parameter values on each row and run them all at once.

Batch processing is ideal when you need to run **one tool many times**, for example a buffer at 10, 100 and 1000 m, or clipping 20 layers to the same boundary.

📖 Tutorial: [Batch processing using the Processing Framework](https://www.qgistutorials.com/en/docs/3/batch_processing.html)

### 5.2 The Graphical Modeler: chaining tools together

When the job needs **several tools in a row**, where the output of one tool is the input to the next, use the **Graphical Modeler** (*Processing → Model Designer*). A model has three kinds of building blocks:

- **inputs** (yellow boxes): layers, numbers, text fields and so on that the user chooses when running the model;
- **algorithms** (white boxes): the processing tools, connected to the inputs and to each other;
- **outputs** (green boxes): the results the model saves.

This is what Tatu's shade model looks like:

<img src="figures/shade_model.png" width="700">

*Figure 4. Tatu's shade model in the QGIS Graphical Modeler. Figure: Tatu Leppämäki.*

Read it from top left to bottom right:

1. **Shade calculation**: r.sun takes the DSM, slope, aspect, day of year, time of day and longitude as inputs.
2. **Fill NoData cells**: cells in shadow come out as NoData, so they are set to −1.
3. **Reclassify**: shaded cells (negative values) become 1, everything else 0.
4. **Sieve**: patches of shade smaller than about 10 m² are removed as noise.
5. **Clip raster by mask layer**: the result is cut to the land area (*Maaraja*), so the sea is not counted as shade.

The output is a clean shade raster (*Varjot*, "shadows"). To get a different moment in time, you only change the *Day of year* and *Time of day* inputs.

A few useful facts about models:

- A model is saved as a **.model3** file, which you can share, or it can be embedded in your QGIS project.
- Once saved, a model appears in the Processing Toolbox under **Models**, and you run it like any other tool. That includes **Run as Batch Process…**: combining a model with batch processing is how you run 48 shade maps without typing parameters 48 times.
- A model can be used inside another model.

📖 QGIS documentation: [The model designer](https://docs.qgis.org/latest/en/docs/user_manual/processing/modeler.html)

### 5.3 From model to code

Models and code are already linked in QGIS. In the Model Designer, you can export any model as a **Python script**. QGIS writes the code for you:

<img src="figures/model_as_python.png" width="520">

*Figure 5. Part of the shade model exported as a Python script. Each input in the model becomes a parameter, and each tool becomes a call to `processing.run(...)`. Figure: Tatu Leppämäki.*

You do **not** need to write code on this course. But it is useful to know what the next step looks like. In the QGIS **Python console** (*Plugins → Python Console*), the buffer example from Section 4 takes only a few lines:

```python
import processing

roads = iface.activeLayer()   # the layer currently selected in the Layers panel

for d in [10, 100, 1000]:
    processing.run("native:buffer", {
        "INPUT": roads,
        "DISTANCE": d,
        "OUTPUT": f"C:/data/roads_buffer_{d}m.gpkg"
    })
```

Note that the buffer distance is in the units of the layer's coordinate reference system. In a projected CRS such as ETRS-TM35FIN (EPSG:3067) that means metres. In a geographic CRS such as WGS 84 it would be degrees, which is almost never what you want.

Python is not limited to QGIS either. Libraries such as *geopandas* let you do the same kind of analysis in scripts and notebooks outside QGIS (see *Going further* below).

### 5.4 Bonus: automating map making with Atlas

Automation is not only for analysis. The **Atlas** function in the QGIS Print Layout produces one map page per feature of a *coverage layer*, for example one map for each district of Helsinki, all with the same layout, legend and scale bar.

📖 Tutorial: [Automating map creation with Print Layout Atlas](https://www.qgistutorials.com/en/docs/3/automating_map_creation.html)

## 6. When should you automate?

Building a model takes time, so it is fair to ask whether it pays off. Tatu's answer: it depends on the task, but quite often you save time in the end. He summed it up in what he called a "highly unscientific automation plot":

<img src="figures/automation_sweet_spot.png" width="460">

*Figure 6. When is automation worth it? The horizontal axis shows how often you will repeat a task, the vertical axis how complex it is. Figure: Tatu Leppämäki.*

Ask yourself two questions:

1. **Is the workflow long or complex?**
2. **Will I repeat it**, later on, with different data or different parameter values?

If the answer to **either** is yes, consider automating. Making a single point map of one dataset is in the corner where clicking through by hand is fine. Creating several buffers around roads is simple, but you repeat it, so it is worth automating. Finding the closest hospital by travel distance for many locations is both complex and repeated. Training and applying a machine learning model on aerial imagery is so complex that you would hardly do it by hand even once.

Even for a one-off task, a model can still be worth it as a record of what you did.

> 💭 **Pause and think**
>
> Think of the tasks you did in Practicals 1–7. Which of them would you place in the "sweet spot"? Is there one you would have liked to automate if you had to repeat it for all municipalities of the Helsinki region?

## 7. This week's practical: day care and air quality

In Practical 8 you will try out process thinking and automation together, using a real planning question:

**Are day care units in Helsinki located too close to busy roads?**

<img src="figures/practical_daycare_map.jpg" width="720">

*Figure 7. Busy roads with buffer zones (orange) and day care units (circles) in the Helsinki region. Red markers show units that fall inside a buffer. The model used is shown in the top left. Figure: Tatu Leppämäki.*

Traffic is a major source of local air pollution, and concentrations are generally highest close to busy roads. To account for this in land use planning, the Helsinki Region Environmental Services Authority (HSY) gives [air quality zones](https://www.hsy.fi/en/air-quality-and-climate/information-for-urban-planning/air-quality-zones/): minimum and recommended distances between roads and housing or *sensitive sites* such as day care centres, schools and hospitals. The busier the road, the larger the distance.

In PPDAC terms, the practical looks roughly like this:

- **Problem:** Which day care units are closer to busy roads than HSY recommends?
- **Plan:** Give each road a buffer whose width depends on its traffic volume, then find the day care units that fall inside a buffer.
- **Data:** Day care units from the City of Helsinki; average weekday traffic volumes on roads from the Finnish Transport Infrastructure Agency; HSY's distance table.
- **Analysis:** A model with three tools: the **Field Calculator** works out the right buffer distance for each road from its traffic volume, the **Variable distance buffer** draws buffers of that width, and **Extract by location** picks out the day care units inside them.
- **Conclusions:** How many units are at risk, and what does that tell us? What are the limitations?

The last point deserves real thought. For example, distance to a road is only a rough proxy for actual exposure, which also depends on wind, buildings and vegetation. And the traffic data covers roads maintained by the state, not every busy city street. You will come back to questions like these in your report.

➡️ Continue to **Practical 8**.

## Check your understanding

Try to answer these before opening the answers.

**1.** What do the letters in PPDAC stand for, and why is it drawn as a cycle?

<details>
<summary>Show answer</summary>

Problem, Plan, Data, Analysis, Conclusions. It is drawn as a cycle because real analyses go back and forth: problems with the data can send you back to the plan, and conclusions often lead to a new problem.

</details>

**2.** You need to clip the same 15 layers to the City of Helsinki boundary. Which QGIS tool is the quickest way to do this?

<details>
<summary>Show answer</summary>

Batch processing: open *Clip*, click *Run as Batch Process…* and add one row per layer. A model is not needed, because only one tool is involved.

</details>

**3.** When does it make more sense to build a model in the Graphical Modeler than to use batch processing?

<details>
<summary>Show answer</summary>

When the job needs several tools in a sequence, where the output of one is the input to the next. You can then also run the whole model as a batch process.

</details>

**4.** Why did Tatu's shade map need automation, even though the analysis worked fine by hand?

<details>
<summary>Show answer</summary>

Each result is valid for only one date and time. A useful answer needed tens of runs, each taking around 30 minutes, with new parameters every time. That is repetitive, slow and easy to get wrong by hand.

</details>

**5.** Your colleague says: "Once the model runs without errors, the results must be correct." What would you answer?

<details>
<summary>Show answer</summary>

Running without errors only means the steps executed. A wrong field, the wrong CRS or unsuitable data would be repeated in every run. Test the model on a small area, check the intermediate outputs, and compare the results with what you know about the place.

</details>

**6.** At which level of automation (Li et al. 2025) is the work you will do in Practical 8?

<details>
<summary>Show answer</summary>

Level 1, routine-aware: you prepare the data and define the workflow, and the computer runs it.

</details>

## Going further

- **Try Tatu's shade map:** [tadusko.github.io/hki_shademap](https://tadusko.github.io/hki_shademap/)
- **Automating GIS Processes** (University of Helsinki): an open online course on doing GIS with Python. It runs every autumn, and no previous coding experience is needed. [autogis-site.readthedocs.io](https://autogis-site.readthedocs.io/en/latest/index.html)
- **Cartographic Visualization in GIS** (University of Helsinki): an open online course on making maps with QGIS and Python. It runs every spring. [cartogis.readthedocs.io](https://cartogis.readthedocs.io/en/latest/)
- **QGIS documentation:** [Model designer](https://docs.qgis.org/latest/en/docs/user_manual/processing/modeler.html) · [Batch processing interface](https://docs.qgis.org/latest/en/docs/user_manual/processing/batch.html)
- **For fun:** xkcd's chart on [how long you can spend automating a routine task before you lose more time than you save](https://xkcd.com/1205/).

## References

- de Smith, M. J., Goodchild, M. F., & Longley, P. A. *Geospatial Analysis: A Comprehensive Guide to Principles, Techniques and Software Tools*. Online edition: [spatialanalysisonline.com](https://www.spatialanalysisonline.com/HTML/index.html?analytical_methodologies.htm)
- Hofierka, J., & Šúri, M. (2002). The solar radiation model for Open source GIS: implementation and applications. *Proceedings of the Open Source GIS – GRASS users conference 2002*, Trento, Italy.
- Helsingin Sanomat (12 June 2025). [Article on the sunniest terraces in central Helsinki](https://www.hs.fi/helsinki/art-2000011290791.html) (in Finnish).
- HSY. [Air quality zones](https://www.hsy.fi/en/air-quality-and-climate/information-for-urban-planning/air-quality-zones/).
- Li, Z., et al. (2025). GIScience in the era of Artificial Intelligence: a research agenda towards Autonomous GIS. *Annals of GIS*. [https://doi.org/10.1080/19475683.2025.2552161](https://doi.org/10.1080/19475683.2025.2552161)
- MacKay, R. J., & Oldford, R. W. (2000). Scientific method, statistical method and the speed of light. *Statistical Science*, 15(3), 254–278. [https://projecteuclid.org/euclid.ss/1009212817](https://projecteuclid.org/euclid.ss/1009212817)